In [ ]:
import os
# if using Apple MPS, fall back to CPU for unsupported ops
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import os

REDIS = os.environ.get('REDIS_SERVICE', "redis") #default=redis
REDIS_PORT = os.environ.get('REDIS_PORT', 6379) #default=6379
redis_max_connections = os.environ.get('REDIS_MAX_CONNECTIONS', 1024) #default=1024
redis_cache_size = os.environ.get('REDIS_CACHE_SIZE', 500)  #default=200
redis_cache_expire_time = os.environ.get('REDIS_CACHE_EXPIRE_TIME', 7*24*60*60)  #default=7*24*60*60
config_input_size = os.environ.get('IMAGE_INPUT_SIZE', 1024)  #default=1024
use_gpu = os.environ.get("SERVER_RUNTIME", "runc") == "nvidia"
user_focus_preload_scope = os.environ.get("PRELOAD_SCOPE", 5) #default scope=5
import time
import pickle
from rds import RedisAccess


In [ ]:
def preload_image_encoder(image,predictor):
    with torch.no_grad():
        image = image.to('cpu').float().unsqueeze(0)
        backbone_out = predictor.forward_image(image)
    return backbone_out,image

In [ ]:
def load_image(dct, predictor, rds, model_id):
    # try:
        #img ==> filestr
        #image_bytes = base64.b64decode(img["data"])
        #npimg = np.fromstring(dct.get('data'), dtype=np.uint8)
        start = time.time()
        #image_key = dct.get('md5')
        image_key = model_id + "_" + dct.get('md5')
        print(f">>>>>>>>>>load image key: {image_key}", flush=True)

        scan_res = rds.redis_scan("seg_image_cache_*")
        image_keys = [i.decode("utf-8") for i in scan_res[1]]
        if len(image_keys) > redis_cache_size:
            # 淘汰掉最老的key
            rds.redis_evict(image_keys)
        if rds.redis_get("seg_image_cache_" + image_key):
            # 用到了的key,刷新过期时间
            rds.redis_expire("seg_image_cache_" + image_key, ex=redis_cache_expire_time)
            print(">>>>>>>>>>Embedding already exists! no need to preload!>>>>>>>>>", flush=True)
            return
        #print("start to preload image!", flush=True)
        #rgb_image = get_rgb_image(npimg)
        rgb_image = dct.get('data')
        # #medical pre process and get image embedding
        # status, error_msg, image_embedding = medical_pre_process_and_get_image_embedding(rgb_image)
        # if not status:
        #     print(error_msg)
        #     return
        # predictor.set_image(rgb_image)
        # image_embedding = predictor.get_image_embedding().cpu().numpy()
        image_embedding,image = preload_image_encoder(rgb_image,predictor)
        image = image.numpy()
        data = {"image_embedding": image_embedding,'image':image}   
        # print(data)
        pickled_data = pickle.dumps(data)
        # print(pickled_data)
        rds.redis_set("seg_image_cache_" + image_key, pickled_data, ex=redis_cache_expire_time)
        print("preload cost = {}".format(time.time() - start), flush=True)
        print(f">>>>>>>>image embedding preload successfully! image_keys: {image_key}", flush=True)
    # except Exception as e:
    #     print("Fail to pre-load image into redis! error={}".format(e), flush=True)
    #     print("KEY {} : Fail to preload!".format(image_key), flush=True)

In [ ]:
from sam2.build_sam import build_sam2_video_predictor

sam2_checkpoint = "/home/a70495/sam2/checkpoints/sam2.1_hiera_base_plus.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_b+.yaml"
device = torch.device("cpu")
predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)

rds = RedisAccess(0)

video_dir = "./videos/bedroom"

# scan all the JPEG frame names in this directory
frame_names = [
    p for p in os.listdir(video_dir)
    if os.path.splitext(p)[-1] in [".jpg", ".jpeg", ".JPG", ".JPEG"]
]
frame_names.sort(key=lambda p: int(os.path.splitext(p)[0]))
inference_state = predictor.init_state(video_path=video_dir)
img_list = [{"md5": os.path.splitext(p)[0], "data": inference_state['images'][idx]} for idx,p in enumerate(frame_names)]
model_id = "sam2.1_hiera_base_plus"
inference_state['rds'] = rds
inference_state['img_list'] = img_list
inference_state['model_id'] = model_id

In [ ]:
def run():
    for idx,img_list_item in enumerate(img_list):
        load_image(img_list_item, predictor, rds, model_id)

from threading import Thread
t = Thread(target=run, args=())
t.start()